# Tests clean maps lib

In [1]:
from random import randint, seed
import torch

In [2]:
import utils.utils_pytorch as utils_pytorch
import utils.clean_map as clean_map

In [3]:
seed(7)

## Outils de génération des données

### Définitions

In [4]:
batch = 3
channel = 3
heigh = 10
width = 10

bc_value = lambda b, c : b * 10 + c
bc_max_value = lambda b, c, cst : bc_value(b, c) * 10 + 7 + cst

def generate_max_and_pos(batch, channel, heigh, width, gen_max=None, one_by=None):
    """
    Retourne tenseur de maximum calculés selon fill_max et leur position aléatoire 3D (batch = 0) ou 4D (batch != 0) par heigh.

    one_by in [None, "tensor", "batch", "channel"]
    """
    pos = []
    max_values = []

    range_b = [randint(0, max(1, batch)-1)] if one_by in ["tensor"] else range(max(1, batch))
    for b in range_b:
        range_c = [randint(0, channel-1)] if one_by in ["tensor", "batch"] else range(channel)
        for c in range_c:
            range_h = [randint(0, heigh-1)] if one_by in ["tensor", "batch", "channel"] else range(heigh)
            for h in range_h:
                pos.append((b, c, h, randint(0, width-1)))
                max_values.append(gen_max(b, c, h) if callable(gen_max) else gen_max)

    pos = torch.tensor(pos)
    max_values = torch.tensor(max_values)

    if batch == 0:
        pos = pos[:, 1:]

    return max_values, pos


def generate_tensor(batch, channel, heigh, width, bc_value, max_values=None, pos=None, dtype=int):
    """
    Retourne un tenseur 3D (batch = 0) ou 4D (batch > 0) remplie selon bc_value, et contenant les max_values à 
    leur position respective indiquée dans pos si fourni.
    """
    assert (max_values!= None and pos != None) or (max_values== None and pos == None), \
        "max_values and pos must be booth not rovided or provided."
    
    t = torch.zeros((max(batch, 1), channel, heigh, width), dtype=dtype)
    for b in range(max(batch, 1)):
        for c in range(channel):
            t[b, c, :, :] = bc_value(b, c) if callable(bc_value) else bc_value

    if b == 0:
        t = t[0]
    
    if max_values != None and pos != None:
        t[*pos.T] = max_values.type(t.dtype)
    
    return t

### Tests des outils

In [5]:
max_values, pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print(max_values, pos, sep="\n")
print(generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=pos))

tensor([ 12,  23,  28, 108, 126, 135, 207, 223, 228])
tensor([[0, 0, 5, 2],
        [0, 1, 6, 0],
        [0, 2, 1, 8],
        [1, 0, 1, 5],
        [1, 1, 9, 0],
        [1, 2, 8, 3],
        [2, 0, 0, 1],
        [2, 1, 6, 6],
        [2, 2, 1, 3]])
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,  12,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
 

In [6]:
max_values, pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="batch")
print(max_values, pos, sep="\n")
print(generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=pos))

tensor([ 15, 116, 216])
tensor([[0, 0, 8, 6],
        [1, 0, 9, 1],
        [2, 0, 9, 0]])
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,  15,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1

In [7]:
max_values, pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print(max_values, pos, sep="\n")
print(generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=pos))

tensor([233])
tensor([[2, 2, 6, 0]])
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1, 

In [8]:
max_values, pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print(max_values, pos, sep="\n")
print(generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=pos))

tensor([10, 25, 31])
tensor([[0, 3, 0],
        [1, 8, 2],
        [2, 4, 6]])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [10,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
 

In [9]:
max_values, pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print(max_values, pos, sep="\n")
print(generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=pos))

tensor([28])
tensor([[2, 1, 9]])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  

## Tests de `cleaning_tensor`

### Sans dimension batch

#### En fournissant des positions

##### Une position par hauteur

In [10]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="height")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([ 7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
        25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36])
> pos :
tensor([[0, 0, 4],
        [0, 1, 8],
        [0, 2, 2],
        [0, 3, 1],
        [0, 4, 9],
        [0, 5, 9],
        [0, 6, 3],
        [0, 7, 5],
        [0, 8, 1],
        [0, 9, 8],
        [1, 0, 1],
        [1, 1, 9],
        [1, 2, 0],
        [1, 3, 9],
        [1, 4, 3],
        [1, 5, 7],
        [1, 6, 8],
        [1, 7, 6],
        [1, 8, 5],
        [1, 9, 7],
        [2, 0, 9],
        [2, 1, 7],
        [2, 2, 5],
        [2, 3, 4],
        [2, 4, 3],
        [2, 5, 2],
        [2, 6, 3],
        [2, 7, 1],
        [2, 8, 9],
        [2, 9, 4]])
t :
tensor([[[ 0,  0,  0,  0,  7,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  8,  0],
         [ 0,  0,  9,  0,  0,  0,  0,  0,  0,  0],
         [ 0, 10,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 11],
        

In [11]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[ 0,  0,  0,  0,  7,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  8,  0],
         [ 0,  0,  9,  0,  0,  0,  0,  0,  0,  0],
         [ 0, 10,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 11],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 12],
         [ 0,  0,  0, 13,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0, 14,  0,  0,  0,  0],
         [ 0, 15,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0, 16,  0]],

        [[ 0, 17,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 18],
         [19,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 20],
         [ 0,  0,  0, 21,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 22,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0, 23,  0],
         [ 0,  0,  0,  0,  0,  0, 24,  0,  0,  0],
         [ 0,  0,  0,  0,  0, 25,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,

In [12]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-1, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(channel, heigh)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[ 7,  8,  9, 10, 11, 12, 13, 14, 15, 16],
        [17, 18, 19, 20, 21, 22, 23, 24, 25, 26],
        [27, 28, 29, 30, 31, 32, 33, 34, 35, 36]])
tensor([[0, 0, 4],
        [0, 1, 8],
        [0, 2, 2],
        [0, 3, 1],
        [0, 4, 9],
        [0, 5, 9],
        [0, 6, 3],
        [0, 7, 5],
        [0, 8, 1],
        [0, 9, 8],
        [1, 0, 1],
        [1, 1, 9],
        [1, 2, 0],
        [1, 3, 9],
        [1, 4, 3],
        [1, 5, 7],
        [1, 6, 8],
        [1, 7, 6],
        [1, 8, 5],
        [1, 9, 7],
        [2, 0, 9],
        [2, 1, 7],
        [2, 2, 5],
        [2, 3, 4],
        [2, 4, 3],
        [2, 5, 2],
        [2, 6, 3],
        [2, 7, 1],
        [2, 8, 9],
        [2, 9, 4]])
OK !!!


##### Une position par canal

In [13]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([15, 22, 31])
> pos :
tensor([[0, 8, 7],
        [1, 5, 7],
        [2, 4, 9]])
t :
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 15,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1, 22,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  

In [14]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 15,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 22,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,

In [15]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([15, 22, 31])
tensor([[0, 8, 7],
        [1, 5, 7],
        [2, 4, 9]])
OK !!!


##### Une seule position pour le tenseur

In [16]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([15])
> pos :
tensor([[0, 8, 6]])
t :
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0, 15,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1, 

In [17]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0, 15,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,

In [18]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([15])
tensor([[0, 8, 6]])
OK !!!


#### En fournissant des valeurs

##### Une valeur par hauteur

In [19]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="height")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([ 7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
        25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36])
> pos :
tensor([[0, 0, 2],
        [0, 1, 5],
        [0, 2, 2],
        [0, 3, 7],
        [0, 4, 6],
        [0, 5, 0],
        [0, 6, 1],
        [0, 7, 8],
        [0, 8, 9],
        [0, 9, 5],
        [1, 0, 5],
        [1, 1, 5],
        [1, 2, 9],
        [1, 3, 7],
        [1, 4, 9],
        [1, 5, 7],
        [1, 6, 1],
        [1, 7, 1],
        [1, 8, 4],
        [1, 9, 7],
        [2, 0, 1],
        [2, 1, 0],
        [2, 2, 4],
        [2, 3, 9],
        [2, 4, 7],
        [2, 5, 4],
        [2, 6, 6],
        [2, 7, 5],
        [2, 8, 0],
        [2, 9, 7]])
t :
tensor([[[ 0,  0,  7,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  8,  0,  0,  0,  0],
         [ 0,  0,  9,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 10,  0,  0],
         [ 0,  0,  0,  0,  0,  0, 11,  0,  0,  0],
        

In [20]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[ 0,  0,  7,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  8,  0,  0,  0,  0],
         [ 0,  0,  9,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 10,  0,  0],
         [ 0,  0,  0,  0,  0,  0, 11,  0,  0,  0],
         [12,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0, 13,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0, 14,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 15],
         [ 0,  0,  0,  0,  0, 16,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0, 17,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0, 18,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 19],
         [ 0,  0,  0,  0,  0,  0,  0, 20,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 21],
         [ 0,  0,  0,  0,  0,  0,  0, 22,  0,  0],
         [ 0, 23,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0, 24,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0, 25,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,

In [21]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-1, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(channel, heigh)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[ 7,  8,  9, 10, 11, 12, 13, 14, 15, 16],
        [17, 18, 19, 20, 21, 22, 23, 24, 25, 26],
        [27, 28, 29, 30, 31, 32, 33, 34, 35, 36]])
tensor([[0, 0, 2],
        [0, 1, 5],
        [0, 2, 2],
        [0, 3, 7],
        [0, 4, 6],
        [0, 5, 0],
        [0, 6, 1],
        [0, 7, 8],
        [0, 8, 9],
        [0, 9, 5],
        [1, 0, 5],
        [1, 1, 5],
        [1, 2, 9],
        [1, 3, 7],
        [1, 4, 9],
        [1, 5, 7],
        [1, 6, 1],
        [1, 7, 1],
        [1, 8, 4],
        [1, 9, 7],
        [2, 0, 1],
        [2, 1, 0],
        [2, 2, 4],
        [2, 3, 9],
        [2, 4, 7],
        [2, 5, 4],
        [2, 6, 6],
        [2, 7, 5],
        [2, 8, 0],
        [2, 9, 7]])
OK !!!


##### Une valeur par canal

In [22]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([12, 26, 34])
> pos :
tensor([[0, 5, 2],
        [1, 9, 1],
        [2, 7, 0]])
t :
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0, 12,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  

In [23]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0, 12,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0, 26,  0,  0,  0,

In [24]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([12, 26, 34])
tensor([[0, 5, 2],
        [1, 9, 1],
        [2, 7, 0]])
OK !!!


##### Une valeur pour le tenseur 

In [25]:
max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([19])
> pos :
tensor([[1, 2, 3]])
t :
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1, 19,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1, 

In [26]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0, 19,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,

In [27]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([19])
tensor([[1, 2, 3]])
OK !!!


### Avec dimension batch

#### En fournissant les positions

##### Une position par hauteur

In [28]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="height")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([  7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,
         21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,
         35,  36, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118,
        119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
        133, 134, 135, 136, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216,
        217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230,
        231, 232, 233, 234, 235, 236])
> pos :
tensor([[0, 0, 0, 6],
        [0, 0, 1, 6],
        [0, 0, 2, 7],
        [0, 0, 3, 1],
        [0, 0, 4, 2],
        [0, 0, 5, 7],
        [0, 0, 6, 6],
        [0, 0, 7, 8],
        [0, 0, 8, 4],
        [0, 0, 9, 2],
        [0, 1, 0, 6],
        [0, 1, 1, 8],
        [0, 1, 2, 4],
        [0, 1, 3, 6],
        [0, 1, 4, 5],
        [0, 1, 5, 6],
        [0, 1, 6, 3],
        [0, 1, 7, 2],
        [0, 1, 8, 1],
        [0, 1, 9, 2],
        [0, 2, 0, 2],
      

In [29]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   7,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   8,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   9,   0,   0],
          [  0,  10,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,  11,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,  12,   0,   0],
          [  0,   0,   0,   0,   0,   0,  13,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,  14,   0],
          [  0,   0,   0,   0,  15,   0,   0,   0,   0,   0],
          [  0,   0,  16,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,  17,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,  18,   0],
          [  0,   0,   0,   0,  19,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,  20,   0,   0,   0],
          [  0,   0,   0,   0,   0,  21,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,  22,   0,   0,   0],
      

In [30]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-1, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch, channel, heigh)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[[  7,   8,   9,  10,  11,  12,  13,  14,  15,  16],
         [ 17,  18,  19,  20,  21,  22,  23,  24,  25,  26],
         [ 27,  28,  29,  30,  31,  32,  33,  34,  35,  36]],

        [[107, 108, 109, 110, 111, 112, 113, 114, 115, 116],
         [117, 118, 119, 120, 121, 122, 123, 124, 125, 126],
         [127, 128, 129, 130, 131, 132, 133, 134, 135, 136]],

        [[207, 208, 209, 210, 211, 212, 213, 214, 215, 216],
         [217, 218, 219, 220, 221, 222, 223, 224, 225, 226],
         [227, 228, 229, 230, 231, 232, 233, 234, 235, 236]]])
tensor([[0, 0, 0, 6],
        [0, 0, 1, 6],
        [0, 0, 2, 7],
        [0, 0, 3, 1],
        [0, 0, 4, 2],
        [0, 0, 5, 7],
        [0, 0, 6, 6],
        [0, 0, 7, 8],
        [0, 0, 8, 4],
        [0, 0, 9, 2],
        [0, 1, 0, 6],
        [0, 1, 1, 8],
        [0, 1, 2, 4],
        [0, 1, 3, 6],
        [0, 1, 4, 5],
        [0, 1, 5, 6],
        [0, 1, 6, 3],
        [0, 1, 7, 2],
        [0, 1, 8, 1],
        [0, 1, 9, 2],
     

##### Une position par canal :

In [31]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([ 11,  19,  27, 115, 119, 127, 211, 221, 232])
> pos :
tensor([[0, 0, 4, 7],
        [0, 1, 2, 8],
        [0, 2, 0, 3],
        [1, 0, 8, 5],
        [1, 1, 2, 8],
        [1, 2, 0, 8],
        [2, 0, 4, 1],
        [2, 1, 4, 8],
        [2, 2, 5, 2]])
t :
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,  11,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1

In [32]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,  11,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,  19,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [33]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch, channel)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[ 11,  19,  27],
        [115, 119, 127],
        [211, 221, 232]])
tensor([[0, 0, 4, 7],
        [0, 1, 2, 8],
        [0, 2, 0, 3],
        [1, 0, 8, 5],
        [1, 1, 2, 8],
        [1, 2, 0, 8],
        [2, 0, 4, 1],
        [2, 1, 4, 8],
        [2, 2, 5, 2]])
OK !!!


##### Une position par batch :

In [34]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="batch")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([ 20, 135, 230])
> pos :
tensor([[0, 1, 3, 8],
        [1, 2, 8, 5],
        [2, 2, 3, 9]])
t :
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,  20,   1],
          

In [35]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,  20,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [36]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([ 20, 135, 230])
tensor([[0, 1, 3, 8],
        [1, 2, 8, 5],
        [2, 2, 3, 9]])
OK !!!


##### Une position pour le tenseur

In [37]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([13])
> pos :
tensor([[0, 0, 6, 3]])
t :
tensor([[[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0, 13,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

         [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
          [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1

In [38]:
cleaned = clean_map.cleaning_tensor(t, pos=max_pos)
print(cleaned)

tensor([[[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0, 13,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

         [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
          

In [39]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-4, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([13])
tensor([[0, 0, 6, 3]])
OK !!!


#### En fournissant des valeurs

##### Une valeur par hauteur

In [40]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="height")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([  7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,
         21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,
         35,  36, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118,
        119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
        133, 134, 135, 136, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216,
        217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230,
        231, 232, 233, 234, 235, 236])
> pos :
tensor([[0, 0, 0, 3],
        [0, 0, 1, 8],
        [0, 0, 2, 7],
        [0, 0, 3, 5],
        [0, 0, 4, 0],
        [0, 0, 5, 0],
        [0, 0, 6, 4],
        [0, 0, 7, 7],
        [0, 0, 8, 4],
        [0, 0, 9, 3],
        [0, 1, 0, 9],
        [0, 1, 1, 5],
        [0, 1, 2, 7],
        [0, 1, 3, 5],
        [0, 1, 4, 5],
        [0, 1, 5, 1],
        [0, 1, 6, 3],
        [0, 1, 7, 1],
        [0, 1, 8, 3],
        [0, 1, 9, 7],
        [0, 2, 0, 3],
      

In [41]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[[  0,   0,   0,   7,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   8,   0],
          [  0,   0,   0,   0,   0,   0,   0,   9,   0,   0],
          [  0,   0,   0,   0,   0,  10,   0,   0,   0,   0],
          [ 11,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [ 12,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,  13,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,  14,   0,   0],
          [  0,   0,   0,   0,  15,   0,   0,   0,   0,   0],
          [  0,   0,   0,  16,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,  17],
          [  0,   0,   0,   0,   0,  18,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,  19,   0,   0],
          [  0,   0,   0,   0,   0,  20,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,  21,   0,   0,   0,   0],
          [  0,  22,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [42]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-1, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch, channel, heigh)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[[  7,   8,   9,  10,  11,  12,  13,  14,  15,  16],
         [ 17,  18,  19,  20,  21,  22,  23,  24,  25,  26],
         [ 27,  28,  29,  30,  31,  32,  33,  34,  35,  36]],

        [[107, 108, 109, 110, 111, 112, 113, 114, 115, 116],
         [117, 118, 119, 120, 121, 122, 123, 124, 125, 126],
         [127, 128, 129, 130, 131, 132, 133, 134, 135, 136]],

        [[207, 208, 209, 210, 211, 212, 213, 214, 215, 216],
         [217, 218, 219, 220, 221, 222, 223, 224, 225, 226],
         [227, 228, 229, 230, 231, 232, 233, 234, 235, 236]]])
tensor([[0, 0, 0, 3],
        [0, 0, 1, 8],
        [0, 0, 2, 7],
        [0, 0, 3, 5],
        [0, 0, 4, 0],
        [0, 0, 5, 0],
        [0, 0, 6, 4],
        [0, 0, 7, 7],
        [0, 0, 8, 4],
        [0, 0, 9, 3],
        [0, 1, 0, 9],
        [0, 1, 1, 5],
        [0, 1, 2, 7],
        [0, 1, 3, 5],
        [0, 1, 4, 5],
        [0, 1, 5, 1],
        [0, 1, 6, 3],
        [0, 1, 7, 1],
        [0, 1, 8, 3],
        [0, 1, 9, 7],
     

##### Une valeur par canal

In [43]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([  7,  19,  27, 109, 124, 128, 207, 225, 235])
> pos :
tensor([[0, 0, 0, 7],
        [0, 1, 2, 9],
        [0, 2, 0, 2],
        [1, 0, 2, 2],
        [1, 1, 7, 9],
        [1, 2, 1, 8],
        [2, 0, 0, 5],
        [2, 1, 8, 8],
        [2, 2, 8, 7]])
t :
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   7,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1

In [44]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   7,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,  19],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [45]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch, channel)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([[  7,  19,  27],
        [109, 124, 128],
        [207, 225, 235]])
tensor([[0, 0, 0, 7],
        [0, 1, 2, 9],
        [0, 2, 0, 2],
        [1, 0, 2, 2],
        [1, 1, 7, 9],
        [1, 2, 1, 8],
        [2, 0, 0, 5],
        [2, 1, 8, 8],
        [2, 2, 8, 7]])
OK !!!


##### Une valeur par batch

In [46]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="batch")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([ 15, 110, 208])
> pos :
tensor([[0, 0, 8, 0],
        [1, 0, 3, 4],
        [2, 0, 1, 8]])
t :
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [ 15,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          

In [47]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [ 15,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [48]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values.view(batch)), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([ 15, 110, 208])
tensor([[0, 0, 8, 0],
        [1, 0, 3, 4],
        [2, 0, 1, 8]])
OK !!!


##### Une valeur pour le tenseur 

In [49]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([127])
> pos :
tensor([[1, 2, 0, 1]])
t :
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
  

In [50]:
cleaned = clean_map.cleaning_tensor(t, values=max_values)
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
      

In [51]:
result_max_values, result_max_pos = utils_pytorch.get_max_by_dim(cleaned, dim=-4, return_pos=True)
print(result_max_values, result_max_pos, sep="\n")
assert torch.equal(result_max_values, max_values), "Error with result_max_values"
assert torch.equal(result_max_pos, max_pos), "Error with result_max_pos"
print("OK !!!")

tensor([127])
tensor([[1, 2, 0, 1]])
OK !!!


## Tests de `cleaning_tensor_after_pooling`

##### Mise au point de l'algo

In [52]:
BATCH_ = 0 # 0
MAX_DIM = -3 # 1 max par élément de t.size(MAX_DIM)

other_max = lambda b, c, h : (1001 + b) if c == 1 else bc_max_value(b, c, h)

#gen_max = bc_max_value
gen_max = other_max

max_values, max_pos = generate_max_and_pos(BATCH_, channel, heigh, width, gen_max=gen_max, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(BATCH_, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

kernel_size=2
stride=1
padding=0
pool_output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
print("> pool_output :", pool_output.size(), pool_output, sep="\n")
print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

> max_values : 
 tensor([  14, 1001,   36])
> pos :
tensor([[0, 7, 5],
        [1, 9, 8],
        [2, 9, 8]])
t :
tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,   14,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   1,    1,    1,    1,    1,    1,    1,    1,    1,    1],
         [   1,    1,    1,    1,    1,    1,    1,    1,    1,    1],
         [   1,    1,    1,    1

Calcul des max de la sortie du pooling si aucune position ou valeur particulière fournie :

In [53]:
# in pool output :
values = None
values_pos = None

if values == None and values_pos == None:
    # Recherche des maximums selon MAX_DIM
    max_values_pool = utils_pytorch.get_max_by_dim(pool_output, dim=MAX_DIM, return_pos=False)
    print("> max_values_pool", max_values_pool, sep="\n")
    
    resize = [1] * len(t.size())
    for d in range(0, len(t.size()) + MAX_DIM):
        resize[d] = pool_output.size(d)
    values = max_values_pool.view(resize)

print("> values", values, sep="\n")

> max_values_pool
tensor([1001])
> values
tensor([[[1001]]])


Recherche des positions des premières occurrences de chaque valeur (transmises ou maximums calculés) dans le tenseur de sortie du pooling dans sa/ses dimension(s) correspondante(s).

In [54]:
if values_pos == None:
    values_pos = utils_pytorch.get_first_occurrence_indices(pool_output, values)
    print(values_pos)
    
    # DEBUG
    temp = torch.zeros_like(pool_output)
    temp[*values_pos.T] = pool_output[*values_pos.T]
    print(temp)

tensor([[1, 8, 7]])
tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
   

Récupération des indices de pooling correspondants (soit avec les maximums ou les valeurs fournies)

In [55]:
values_pool_indices = torch.full_like(pool_indices, -1) # -1 car ce n'est pas une valeur d'indice de pooling
values_pool_indices[*values_pos.T] = pool_indices[*values_pos.T]
print(values_pool_indices)

tensor([[[-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1]],

        [[-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, 98, -1]],

        [[-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         [-1, -1, -1, -1, -1, -1, -1, -1, -1],
         

Construction du masque permettant d'obtenir un tenseur ne contenant QUE les indices de pooling concernés : génératio du tenseur nettoyé "tous".

In [56]:
values_pool_indices_vector = \
    utils_pytorch.get_max_by_dim(values_pool_indices, dim=MAX_DIM+1, return_pos=False)
print(values_pool_indices_vector)

resize = [1] * len(pool_output.size())
for d in range(0, len(pool_output.size()) + MAX_DIM+1):
    resize[d] = pool_output.size(d)

values_pool_indices_vector_resized = values_pool_indices_vector.view(resize)
print(values_pool_indices_vector_resized)

mask_values_pool_indices = (pool_indices == values_pool_indices_vector_resized)
print(mask_values_pool_indices)
cleaned = pool_output * mask_values_pool_indices

print(cleaned)

tensor([-1, 98, -1])
tensor([[[-1]],

        [[98]],

        [[-1]]])
tensor([[[False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False]],

        [[False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False],
         [False, False, False, False, 

In [57]:
### TEST
pos_values = values_pos
print("> pos_values", pos_values, sep="\n")
cleaned = clean_map.cleaning_tensor_after_pooling(
    #pool_output, pos=pos_values, pool_indices=pool_indices, max_dim=MAX_DIM, keep_only_last_occurrence=False
    pool_output, pos=pos_values, pool_indices=pool_indices, keep_only_last_occurrence=False
    )
print("> cleaned", cleaned, sep="\n")

#_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=MAX_DIM, return_pos=True)
#expected = pool_max_pos
#print(pos_max_cleaned)

#assert torch.equal(expected, pos_max_cleaned), "Error"
#print("OK !!!")

> pos_values
tensor([[1, 8, 7]])
> cleaned
tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,

Ou bien seulement les dernières occurences

In [58]:
last_occurrence_pool_indices = utils_pytorch.get_last_occurrence_indices(
    cleaned, values
)

cleaned = torch.zeros_like(pool_output)
cleaned[*last_occurrence_pool_indices.T] = pool_output[*last_occurrence_pool_indices.T]
print(cleaned)

tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,  

### Sans dimension batch

#### Une position par canal

In [59]:
verbose = True

max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
if verbose:
    print("> max_values :", max_values, sep="\n")
    print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
if verbose :
    print("> t :", t.size(), t, sep="\n")

kernel_size=2
stride=1
padding=0
output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
if verbose:
    print("> output :", output.size(), output, sep="\n")
    print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

# récupération du maximum de chaque canal en un vecteur simple
pool_max_values, pool_max_pos = utils_pytorch.get_max_by_dim(output, dim=-2, return_pos=True)
if verbose:
    print("> pool_max_values :", pool_max_values.size(), pool_max_values, sep="\n")
    print("> pool_max_pos :", pool_max_pos.size(), pool_max_pos, sep="\n")
    for i, pos in enumerate(pool_max_pos):
        print(f"{i:2d} : {pos}")

> max_values :
tensor([10, 24, 35])
> pos :
tensor([[0, 3, 4],
        [1, 7, 8],
        [2, 8, 7]])
> t :
torch.Size([3, 10, 10])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0, 10,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],

##### En fournissant des positions

En préservant toutes les positions de la valeur correspondante utiles pour l'unpooling

In [60]:
pos_values = pool_max_pos[[1, 6, 10]]
print(pos_values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    output, pos=pos_values, pool_indices=pool_indices, keep_only_last_occurrence=False
    )
print("cleaned", cleaned, sep="\n")
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
expected = pool_max_pos
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[0, 2, 4],
        [1, 7, 7],
        [2, 8, 6]])
cleaned
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0, 10, 10,  0,  0,  0,  0],
         [ 0,  0,  0, 10, 10,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 24, 24],
         [ 0,  0,  0,  0,  0,  0,  0, 24, 24],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,

en préservant la dernière position de la valeur correspondante utile pour l'unpooling

In [61]:
pos_values = pool_max_pos[[1, 6, 10]]
print(pos_values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    output, pos=pos_values, pool_indices=pool_indices, keep_only_last_occurrence=True
    )
print("cleaned", cleaned, sep="\n")
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
expected = pool_max_pos[[3, 7, 11]]
print(expected)
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[0, 2, 4],
        [1, 7, 7],
        [2, 8, 6]])
cleaned
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0, 10,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0, 24],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,

##### En fournissant des valeurs

In [62]:
values = pool_max_values.view(3, 1, 1)
print(values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    output, values=values, pool_indices=pool_indices, keep_only_last_occurrence=False
    )
print("cleaned", cleaned, sep="\n")
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-2, return_pos=True)
expected = pool_max_pos
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[[10]],

        [[24]],

        [[35]]])
cleaned
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0, 10, 10,  0,  0,  0,  0],
         [ 0,  0,  0, 10, 10,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0, 24, 24],
         [ 0,  0,  0,  0,  0,  0,  0, 24, 24],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0

#### Une position pour le tenseur

In [63]:
verbose = True

max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="tensor")
if verbose:
    print("> max_values :", max_values, sep="\n")
    print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
if verbose :
    print("> t :", t.size(), t, sep="\n")

kernel_size=2
stride=1
padding=0
output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
if verbose:
    print("> output :", output.size(), output, sep="\n")
    print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

# récupération du maximum de chaque canal en un vecteur simple
pool_max_values, pool_max_pos = utils_pytorch.get_max_by_dim(output, dim=-3, return_pos=True)
if verbose:
    print("> pool_max_values :", pool_max_values.size(), pool_max_values, sep="\n")
    print("> pool_max_pos :", pool_max_pos.size(), pool_max_pos, sep="\n")
    for i, pos in enumerate(pool_max_pos):
        print(f"{i:2d} : {pos}")

> max_values :
tensor([35])
> pos :
tensor([[2, 8, 4]])
> t :
torch.Size([3, 10, 10])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,

##### En fournissant des positions

En préservant toutes les positions de la valeur correspondante utiles pour l'unpooling

In [64]:
pos_values = pool_max_pos[[0]]
print(pos_values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    #output, pos=pos_values, pool_indices=pool_indices, max_dim=-3, keep_only_last_occurrence=False
    output, pos=pos_values, pool_indices=pool_indices, keep_only_last_occurrence=False
    )
print("cleaned", cleaned, sep="\n")
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
expected = pool_max_pos
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[2, 7, 3]])
cleaned
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0

en préservant la dernière position de la valeur correspondante utile pour l'unpooling

In [65]:
pos_values = pool_max_pos[[0]]
print(pos_values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    output, pos=pos_values, pool_indices=pool_indices, keep_only_last_occurrence=True
    )
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
expected = pool_max_pos[[-1]]
print(expected)
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[2, 7, 3]])
tensor([[2, 8, 4]])
tensor([[2, 8, 4]])
OK !!!


##### En fournissant des valeurs

### Avec dimension batch

#### Une position par canal

In [66]:
verbose = True

max_values, max_pos = generate_max_and_pos(0, channel, heigh, width, gen_max=bc_max_value, one_by="channel")
if verbose:
    print("> max_values :", max_values, sep="\n")
    print("> pos :", max_pos, sep="\n")

t = generate_tensor(0, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
if verbose :
    print("> t :", t.size(), t, sep="\n")

kernel_size=2
stride=1
padding=0
output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
if verbose:
    print("> output :", output.size(), output, sep="\n")
    print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

# récupération du maximum de chaque canal en un vecteur simple
pool_max_values, pool_max_pos = utils_pytorch.get_max_by_dim(output, dim=-2, return_pos=True)
if verbose:
    print("> pool_max_values :", pool_max_values.size(), pool_max_values, sep="\n")
    print("> pool_max_pos :", pool_max_pos.size(), pool_max_pos, sep="\n")
    for i, pos in enumerate(pool_max_pos):
        print(f"{i:2d} : {pos}")

> max_values :
tensor([15, 24, 33])
> pos :
tensor([[0, 8, 3],
        [1, 7, 2],
        [2, 6, 1]])
> t :
torch.Size([3, 10, 10])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0, 15,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
         [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],

##### En fournissant des positions

En préservant toutes les positions de la valeur correspondante utiles pour l'unpooling

##### En fournissant des valeurs

#### Une position par batch

In [67]:
verbose = True

max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value, one_by="batch")
if verbose:
    print("> max_values :", max_values, sep="\n")
    print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
if verbose :
    print("> t :", t.size(), t, sep="\n")

kernel_size=2
stride=1
padding=0
pool_output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
if verbose:
    print("> pool_output :", pool_output.size(), pool_output, sep="\n")
    print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

# récupération du maximum de chaque canal en un vecteur simple
pool_max_values, pool_max_pos = utils_pytorch.get_max_by_dim(pool_output, dim=-3, return_pos=True)
if verbose:
    print("> pool_max_values :", pool_max_values.size(), pool_max_values, sep="\n")
    print("> pool_max_pos :", pool_max_pos.size(), pool_max_pos, sep="\n")
    for i, pos in enumerate(pool_max_pos):
        print(f"{i:2d} : {pos}")

> max_values :
tensor([ 24, 110, 210])
> pos :
tensor([[0, 1, 7, 5],
        [1, 0, 3, 6],
        [2, 0, 3, 4]])
> t :
torch.Size([3, 3, 10, 10])
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
          [  1,   1,   1,   1,   1,   1,   1,

##### En fournissant des valeurs

En l'occurrence, un maximum par batch

In [68]:
values = pool_max_values.view(batch, 1, 1, 1)
print(values)
cleaned = clean_map.cleaning_tensor_after_pooling(
    tensor=pool_output, values=values, pool_indices=pool_indices, keep_only_last_occurrence=False
    )
print("cleaned", cleaned, sep="\n")
_, pos_max_cleaned = utils_pytorch.get_max_by_dim(cleaned, dim=-3, return_pos=True)
expected = pool_max_pos
print(pos_max_cleaned)

assert torch.equal(expected, pos_max_cleaned), "Error"
print("OK !!!")

tensor([[[[ 24]]],


        [[[110]]],


        [[[210]]]])
cleaned
tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,  24,  24,   0,   0,   0],
          [  0, 

## Test de `cleaning`

### Avec dimension batch

In [69]:
BATCH_ = batch # 0 or batch
MAX_DIM = -3 # 1 max par élément de t.size(MAX_DIM)

other_max = lambda b, c, h : (1001 + b) if c == 1 else bc_max_value(b, c, h)

#gen_max = bc_max_value
gen_max = other_max

max_values, max_pos = generate_max_and_pos(BATCH_, channel, heigh, width, gen_max=gen_max, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(BATCH_, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

kernel_size=2
stride=1
padding=0
pool_output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
print("> pool_output :", pool_output.size(), pool_output, sep="\n")
print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

> max_values : 
 tensor([   8, 1001,   31,  114, 1002,  134,  210, 1003,  233])
> pos :
tensor([[0, 0, 1, 2],
        [0, 1, 5, 2],
        [0, 2, 4, 2],
        [1, 0, 7, 3],
        [1, 1, 1, 6],
        [1, 2, 7, 2],
        [2, 0, 3, 2],
        [2, 1, 6, 8],
        [2, 2, 6, 5]])
t :
tensor([[[[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    8,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,   

Une valeur max par batch :

In [70]:
result = clean_map.cleaning(pool_output, pos=None, pool_indices=pool_indices, max_dim=MAX_DIM)
print(result)

tensor([[[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

         [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0, 1001, 1001,    0,    0,    0,    0,    0,    0],
          [   0, 1001, 1001,    0,    0,    0,    0,    0,    0],
        

In [71]:
result = clean_map.cleaning(pool_output, pos=None, pool_indices=pool_indices, max_dim=MAX_DIM, return_pos=True)
print(result)
print(pool_output[*result[1].T])

(tensor([[[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

         [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0, 1001, 1001,    0,    0,    0,    0,    0,    0],
          [   0, 1001, 1001,    0,    0,    0,    0,    0,    0],
       

## Test de `clean_feature_maps`

Différents cas :
- idx_map == None and pos == None
- idx_map == None and pos != None
- idx_map != None and pos == None
- idx_map != None and pos != None

avec ou sans la dimension `batch`

### Sans Batch

In [72]:
BATCH_ = 0 # 0 or batch
MAX_DIM = -3 # 1 max par élément de t.size(MAX_DIM)

other_max = lambda b, c, h : (1001 + b) if c == 1 else bc_max_value(b, c, h)

#gen_max = bc_max_value
gen_max = other_max

max_values, max_pos = generate_max_and_pos(BATCH_, channel, heigh, width, gen_max=gen_max, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(BATCH_, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

kernel_size=2
stride=1
padding=0
pool_output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
print("> pool_output :", pool_output.size(), pool_output, sep="\n")
print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

> max_values : 
 tensor([  13, 1001,   28])
> pos :
tensor([[0, 6, 3],
        [1, 5, 5],
        [2, 1, 5]])
t :
tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,   13,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   1,    1,    1,    1,    1,    1,    1,    1,    1,    1],
         [   1,    1,    1,    1,    1,    1,    1,    1,    1,    1],
         [   1,    1,    1,    1

Nettoyage en gardant un max pour le tenseur :

In [73]:
result = clean_map.clean_feature_maps(pool_output, idx_map=False, pos=None, pool_indices=pool_indices)
print(result)

tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0,    0,  

Nettoyage en gardant un max par canal :

In [74]:
result = clean_map.clean_feature_maps(pool_output, idx_map=None, pos=None, pool_indices=pool_indices)
print(result)

tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,   13,   13,    0,    0,    0,    0,    0],
         [   0,    0,   13,   13,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0,    0,  

Nettoyage en préservant le maximum du canal indiqué (carte) :

In [75]:
result = clean_map.clean_feature_maps(pool_output, idx_map=2, pos=None, pool_indices=pool_indices)
print(result)

tensor([[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

        [[ 0,  0,  0,  0, 28, 28,  0,  0,  0],
         [ 0,  0,  0,  0, 28, 28,  0,  0,  0],
         [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
         

Nettoyage en préservant la valeur position selon idx_map, (posH, posW) :

In [76]:
pos = (5, 5)
print(pos)
result = clean_map.clean_feature_maps(pool_output, idx_map=2, pos=pos, pool_indices=pool_indices)
print(result)

(5, 5)
tensor([[[0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 2, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0],
 

Nettoyage en préservant la valeur position (posH, posW) pour chaque canal :

In [77]:
pos = (5, 5)
print(pos)
result = clean_map.clean_feature_maps(pool_output, idx_map=None, pos=pos, pool_indices=pool_indices)
print(result)

(5, 5)
tensor([[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

        [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0,    0,    0,    0, 1001, 1001,    0,    0,    0],
         [   0, 

### Avec batch

In [78]:
BATCH_ = batch # 0 or batch
MAX_DIM = -3 # 1 max par élément de t.size(MAX_DIM)

other_max = lambda b, c, h : (1001 + b) if c == 1 else bc_max_value(b, c, h)

#gen_max = bc_max_value
gen_max = other_max

max_values, max_pos = generate_max_and_pos(BATCH_, channel, heigh, width, gen_max=gen_max, one_by="channel")
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(BATCH_, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

kernel_size=2
stride=1
padding=0
pool_output, pool_indices = torch.nn.functional.max_pool2d(
    t, kernel_size=kernel_size, stride=stride, padding=padding, return_indices=True
    )
print("> pool_output :", pool_output.size(), pool_output, sep="\n")
print("> pool_indices :", pool_indices.size(), pool_indices, sep="\n")

> max_values : 
 tensor([   7, 1001,   34,  113, 1002,  131,  208, 1003,  228])
> pos :
tensor([[0, 0, 0, 5],
        [0, 1, 8, 7],
        [0, 2, 7, 0],
        [1, 0, 6, 5],
        [1, 1, 8, 9],
        [1, 2, 4, 8],
        [2, 0, 1, 1],
        [2, 1, 3, 1],
        [2, 2, 1, 4]])
t :
tensor([[[[   0,    0,    0,    0,    0,    7,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,   

Calcul des max de la sortie du pooling si aucune position ou valeur particulière fournie :

In [79]:
# in pool output :
values = None
values_pos = None

if values == None and values_pos == None:
    # Recherche des maximums selon MAX_DIM
    max_values_pool, max_values_pool_pos = utils_pytorch.get_max_by_dim(pool_output, dim=MAX_DIM, return_pos=True)
    print("> max_values_pool", max_values_pool, sep="\n")
    print("> max_values_pool_pos", max_values_pool_pos, sep="\n")
    
    resize = [1] * len(t.size())
    for d in range(0, len(t.size()) + MAX_DIM):
        resize[d] = pool_output.size(d)
    values = max_values_pool.view(resize)

print("> values", values, sep="\n")

> max_values_pool
tensor([1001, 1002, 1003])
> max_values_pool_pos
tensor([[0, 1, 7, 6],
        [0, 1, 7, 7],
        [0, 1, 8, 6],
        [0, 1, 8, 7],
        [1, 1, 7, 8],
        [1, 1, 8, 8],
        [2, 1, 2, 0],
        [2, 1, 2, 1],
        [2, 1, 3, 0],
        [2, 1, 3, 1]])
> values
tensor([[[[1001]]],


        [[[1002]]],


        [[[1003]]]])


Nettoyage en gardant un max par batch :

In [91]:
result = clean_map.clean_feature_maps(pool_output, idx_map=False, pos=None, pool_indices=pool_indices, return_pos=True)
print(result)

(tensor([[[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

         [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
       

Nettoyage en préservant le max de chaque canal, par batch :

In [92]:
result = clean_map.clean_feature_maps(pool_output, idx_map=None, pos=None, pool_indices=pool_indices, return_pos=True)
print(result)

(tensor([[[[   0,    0,    0,    0,    7,    7,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

         [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
       

Nettoyage en préservant le maximum du canal indiqué (carte), par batch :

In [82]:
result = clean_map.clean_feature_maps(pool_output, idx_map=2, pos=None, pool_indices=pool_indices)
print(result)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0

Nettoyage en préservant la valeur position selon idx_map, (posH, posW), pour chaque groupe :

In [93]:
pos = max_values_pool_pos[2][2:]
print(pos)
result = clean_map.clean_feature_maps(pool_output, idx_map=2, pos=pos, pool_indices=pool_indices, return_pos=True)
print(result)

tensor([8, 6])
(tensor([[[[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

         [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0]],

         [[ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  0,  0,  0,  0,  0,  0,  0],
          [ 0,  0,  

Nettoyage en préservant la valeur position (posH, posW) pour chaque canal, pour chaque batch :

In [94]:
pos = max_values_pool_pos[2][2:]
print(pos)
result = clean_map.clean_feature_maps(pool_output, idx_map=None, pos=pos, pool_indices=pool_indices, return_pos=True)
print(result)

tensor([8, 6])
(tensor([[[[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0]],

         [[   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,    0,    0,    0,    0,    0,    0,    0,

### Mise au point de l'algo

In [85]:
idx_map = 2
max_ = utils_pytorch.get_max_by_dim(pool_output, dim=-2, return_pos=False)
print(max_)
# On ne garde que les valeurs des canaux idx_map <=> les autres sont mis à feature_maps.min() - 1
idx_map_max = torch.full((max_.numel(),), pool_output.min() - 1)
idx_map_max[idx_map::pool_output.size(-3)] = max_.flatten()[idx_map::pool_output.size(-3)]

max_dim = -2
idx_map_max = utils_pytorch.partial_resize_like(idx_map_max, pool_output, max_dim)


print(idx_map_max)

tensor([[   7, 1001,   34],
        [ 113, 1002,  131],
        [ 208, 1003,  228]])
tensor([[[[ -1]],

         [[ -1]],

         [[ 34]]],


        [[[ -1]],

         [[ -1]],

         [[131]]],


        [[[ -1]],

         [[ -1]],

         [[228]]]])


In [86]:
pos = utils_pytorch.get_first_occurrence_indices(pool_output, idx_map_max)
print(pos)

tensor([[0, 2, 6, 0],
        [1, 2, 3, 7],
        [2, 2, 0, 3]])


In [87]:
values_pool_indices = torch.full_like(pool_indices, -1) # -1 car ce n'est pas une valeur d'indice de pooling
values_pool_indices[*pos.T] = pool_indices[*pos.T]
print(values_pool_indices)

tensor([[[[-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1]],

         [[-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1]],

         [[-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -

In [88]:
MAX_DIM = -2
values_pool_indices_vector = utils_pytorch.get_max_by_dim(
    values_pool_indices, dim=MAX_DIM, return_pos=False
)
print(values_pool_indices_vector)

tensor([[-1, -1, 70],
        [-1, -1, 48],
        [-1, -1, 14]])


In [89]:
values_pool_indices_vector_resized = utils_pytorch.partial_resize_like(values_pool_indices_vector, pool_output, MAX_DIM)
print(values_pool_indices_vector_resized)

tensor([[[[-1]],

         [[-1]],

         [[70]]],


        [[[-1]],

         [[-1]],

         [[48]]],


        [[[-1]],

         [[-1]],

         [[14]]]])


In [90]:
mask_values_pool_indices = (pool_indices == values_pool_indices_vector_resized)
cleaned = pool_output * mask_values_pool_indices
print(cleaned)

tensor([[[[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0]],

         [[  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0,   0,   0,   0,   0,   0],
          [  0,   0,   0,   0